# MedVision Agent â€” Brain Tumor U-Net (BraTS)

> âš ï¸ **This is a research prototype and not approved for clinical diagnosis.**
> Train and evaluate on public, properly licensed, de-identified datasets only.

**Goal:** train a tiny 2-D U-Net on axial brain-MRI slices (FLAIR) to segment tumor
tissue, then export `brain_unet.onnx` for the inference service.

- Dataset: **BraTS** (any edition with `{case}_flair.nii.gz` + `{case}_seg.nii.gz`).
  Kaggle mirror example: `awsaf49/brats20-dataset-training-validation`. Official
  distribution via Synapse. **Review the dataset license before use.**
- Approach: single-slice 2-D segmentation (deliberately lightweight â€” no 3-D).
- Loss: Dice + BCE Â· Metrics: Dice / IoU / precision / recall.
- Runs on a free Colab/Kaggle T4 in well under an hour for the teaching subset.

In [ ]:
%pip install -q nibabel onnx onnxruntime matplotlib tqdm
# torch + torchvision are preinstalled on Colab/Kaggle GPU runtimes

In [ ]:
import json, math, random, shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

CONFIG = {
    'SIZE': 256,        # matches inference-service input contract
    'BATCH': 16,
    'EPOCHS': 25,
    'LR': 1e-3,
    'POS_FRAC': 0.5,    # keep this share of slices tumor-bearing
    'MAX_CASES': 40,    # RAM-friendly teaching subset; raise if you like
    'SEED': 42,
}
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
print('device:', DEVICE)

## 1 Â· Locate BraTS data

Point `DATA_ROOT` at a folder whose subdirectories look like:
`BraTS2021_00000/BraTS2021_00000_flair.nii.gz` and `..._seg.nii.gz`
(the glob below also handles the BraTS2020 naming `*_seg.nii`).

In [ ]:
# TODO(student): set DATA_ROOT to your local BraTS path or mount Kaggle input.
CANDIDATES = [
    '/kaggle/input/brats20-dataset-training-validation',
    '/content/BraTS2021',
    './data/brats',
]
DATA_ROOT = next((p for p in CANDIDATES if Path(p).exists()), None)
assert DATA_ROOT, (
    'BraTS data not found. Download it (respecting its license) and set '
    'DATA_ROOT manually, e.g. DATA_ROOT = "/path/to/brats".'
)

case_dirs = sorted({str(p.parent) for p in Path(DATA_ROOT).rglob('*seg*.nii*')})
random.Random(CONFIG['SEED']).shuffle(case_dirs)
case_dirs = case_dirs[: CONFIG['MAX_CASES']]
print(f'{len(case_dirs)} cases located under {DATA_ROOT}')
assert case_dirs, 'No case directories matched *seg*.nii*'

## 2 Â· Slice dataset

Key teaching points:
- Split by **case**, never by slice (prevents leakage between train/val).
- Balance positive (tumor) vs negative slices at index-build time (`POS_FRAC`).
- Per-volume z-score normalization after percentile clipping.

In [ ]:
import nibabel as nib

class BrainSliceDataset(Dataset):
    '''Axial FLAIR slices labeled tumor(=any seg label>0) vs background.'''

    def __init__(self, case_dirs, size=256, pos_frac=0.5, augment=False):
        self.size, self.augment = size, augment
        self._cache = {}
        self.index = []  # (case_dir, z, has_tumor)
        per_case_pos = {}
        for cd in case_dirs:
            seg_p = next(Path(cd).glob('*seg*.nii*'), None)
            fl_p = next(Path(cd).glob('*_flair.nii.gz'), None)
            if not (seg_p and fl_p):
                continue
            seg = np.asanyarray(nib.load(str(seg_p)).dataobj)
            pos = sorted(int(z) for z in np.unique(np.nonzero(seg)[2]))
            neg_all = [z for z in range(seg.shape[2]) if z not in set(pos)]
            rng = random.Random((hash(cd) ^ 0xB5A7) & 0xFFFF)
            k_neg = min(int(len(pos) * (1 - pos_frac) / max(pos_frac, 1e-6)), len(neg_all))
            for z in pos:
                self.index.append((cd, z, True))
            for z in rng.sample(neg_all, k_neg):
                self.index.append((cd, z, False))
            per_case_pos[cd] = len(pos)
        self.case_dirs = list(per_case_pos.keys())

    def _load_volume(self, cd):
        if cd not in self._cache:
            fl = next(Path(cd).glob('*_flair.nii.gz'))
            sg = next(Path(cd).glob('*seg*.nii*'))
            img = np.asanyarray(nib.load(str(fl)).dataobj, dtype=np.float32)
            msk = (np.asanyarray(nib.load(str(sg)).dataobj) > 0).astype(np.float32)
            self._cache[cd] = (img, msk)  # NOTE: caches in RAM; lower MAX_CASES if OOM
        return self._cache[cd]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        cd, z, _ = self.index[i]
        img, msk = self._load_volume(cd)
        x, y = img[:, :, z], msk[:, :, z]

        # robust normalization â†’ [0,1]
        lo, hi = np.percentile(x, (0.5, 99.5))
            span = max(hi - lo, 1e-6)
            x = (np.clip(x, lo, hi) - lo) / span

        if self.augment:
            if random.random() < 0.5:
                x, y = x[:, ::-1].copy(), y[:, ::-1].copy()
            if random.random() < 0.5:
                x, y = x[::-1, :].copy(), y[::-1, :].copy()
            k = random.randint(0, 3)
            if k:
                x, y = np.rot90(x, k).copy(), np.rot90(y, k).copy()
            x = x * np.random.uniform(0.9, 1.1)

        x = np.array(Image.fromarray(np.clip(x * 255, 0, 255).astype(np.uint8), 'L')
                     .resize((self.size, self.size), Image.BILINEAR), dtype=np.float32) / 255.0
        y = np.array(Image.fromarray((y * 255).astype(np.uint8), 'L')
                     .resize((self.size, self.size), Image.NEAREST), dtype=np.float32) / 255.0
        return torch.from_numpy(x[None]).float(), (torch.from_numpy(y[None]).float() > 0.5).float()

# case-level split (no slice leakage)
split = int(len(case_dirs) * 0.85)
train_ds = BrainSliceDataset(case_dirs[:split], CONFIG['SIZE'], CONFIG['POS_FRAC'], augment=True)
val_ds   = BrainSliceDataset(case_dirs[split:], CONFIG['SIZE'], 1.0, augment=False)
train_loader = DataLoader(train_ds, batch_size=CONFIG['BATCH'], shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds, batch_size=CONFIG['BATCH'], shuffle=False, num_workers=2, pin_memory=True)
pos_share = sum(1 for _,_,p in train_ds.index if p) / max(len(train_ds), 1)
print(f'train slices: {len(train_ds)} ({pos_share:.0%} tumor) | val cases: {len(val_ds.case_dirs)}')

In [ ]:
# sanity-check a few training samples
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for j in range(4):
    x, y = train_ds[random.randrange(len(train_ds))]
    axes[0, j].imshow(x[0], cmap='gray'); axes[0, j].set_title('image'); axes[0, j].axis('off')
    axes[1, j].imshow(x[0], cmap='gray'); axes[1, j].imshow(y[0], alpha=0.4, cmap='Reds'); axes[1, j].axis('off')
plt.tight_layout(); plt.show()

## 3 Â· Model, loss, metrics

`MiniUNet` (~0.2 M params at base=16) is intentionally small so the exported ONNX
runs comfortably inside free-tier CPU quotas.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, cin, cout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
            nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class MiniUNet(nn.Module):
    def __init__(self, base=16):
        super().__init__()
        self.d1, self.d2, self.d3 = DoubleConv(1, base), DoubleConv(base, base*2), DoubleConv(base*2, base*4)
        self.pool = nn.MaxPool2d(2)
        self.u2, self.u1 = nn.ConvTranspose2d(base*4, base*2, 2, stride=2), nn.ConvTranspose2d(base*2, base, 2, stride=2)
        self.c2, self.c1 = DoubleConv(base*4, base*2), DoubleConv(base*2, base)
        self.out = nn.Conv2d(base, 1, 1)
    def forward(self, x):
        s1 = self.d1(x); s2 = self.d2(self.pool(s1)); b = self.d3(self.pool(s2))
        u = self.c2(torch.cat([self.u2(b), s2], dim=1))
        u = self.c1(torch.cat([self.u1(u), s1], dim=1))
        return self.out(u)  # raw logits

model = MiniUNet().to(DEVICE)
print(f'{sum(p.numel() for p in model.parameters())/1e6:.2f} M parameters')

In [ ]:
class DiceBCELoss(nn.Module):
    '''BCE stabilizes early training; soft-Dice optimizes the reported metric.'''
    def __init__(self, dice_weight=0.5, smooth=1.0):
        super().__init__()
        self.bce, self.w, self.smooth = nn.BCEWithLogitsLoss(), dice_weight, smooth
    def forward(self, logits, target):
        bce = self.bce(logits, target)
        prob = torch.sigmoid(logits)
        inter = (prob * target).sum(dim=(2, 3))
        union = prob.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice = 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()
        return bce + self.w * dice

@torch.no_grad()
def segmentation_metrics(logits, target, thr=0.5, eps=1e-7):
    '''Dataset-level binary segmentation metrics (batch-aggregated).'''
    pred = (torch.sigmoid(logits) > thr).float()
    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()
    return {
        'dice': float((2 * tp + eps) / (2 * tp + fp + fn + eps)),
        'iou': float((tp + eps) / (tp + fp + fn + eps)),
        'precision': float((tp + eps) / (tp + fp + eps)),
        'recall': float((tp + eps) / (tp + fn + eps)),
    }

criterion = DiceBCELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['LR'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == 'cuda'))

## 4 Â· Training loop

In [ ]:
def train_one_epoch(loader):
    model.train(); total = 0.0
    for img, msk in loader:
        img, msk = img.to(DEVICE), msk.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            loss = criterion(model(img), msk)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total += float(loss) * img.size(0)
    return total / max(1, len(loader.dataset))

@torch.no_grad()
def evaluate(loader):
    model.eval(); total = 0.0; agg = {'dice': [], 'iou': [], 'precision': [], 'recall': []}
    for img, msk in loader:
        img, msk = img.to(DEVICE), msk.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=(DEVICE == 'cuda')):
            logits = model(img)
        total += float(criterion(logits.float(), msk)) * img.size(0)
        for k, v in segmentation_metrics(logits.float(), msk).items():
            agg[k].append(v)
    out = {k: sum(v) / len(v) for k, v in agg.items()}
    out['loss'] = total / max(1, len(loader.dataset))
    return out

In [ ]:
history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_iou': []}
best_dice, best_path = -1.0, OUT_DIR / 'brain_unet_best.pth'

for epoch in range(1, CONFIG['EPOCHS'] + 1):
    tl = train_one_epoch(train_loader)
    ev = evaluate(val_loader)
    scheduler.step(ev['dice'])
    history['train_loss'].append(tl); history['val_loss'].append(ev['loss'])
    history['val_dice'].append(ev['dice']); history['val_iou'].append(ev['iou'])
    star = ''
    if ev['dice'] > best_dice:
        best_dice = ev['dice']
        torch.save(model.state_dict(), best_path)  # best .pth checkpoint
        star = '  â† saved'
    print(f"epoch {epoch:02d} | train_loss {tl:.4f} | val_loss {ev['loss']:.4f} | "
          f"dice {ev['dice']:.4f} | iou {ev['iou']:.4f} | prec {ev['precision']:.4f} | rec {ev['recall']:.4f}{star}")

print(f'best val dice: {best_dice:.4f}')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history['train_loss'], label='train'); ax[0].plot(history['val_loss'], label='val')
ax[0].set_title('Dice+BCE loss'); ax[0].legend()
ax[1].plot(history['val_dice'], label='dice'); ax[1].plot(history['val_iou'], label='iou')
ax[1].set_title('validation overlap metrics'); ax[1].legend()
plt.tight_layout(); plt.show()

## 5 Â· Export to ONNX + parity check

The service expects: input `input` = `1Ã—1Ã—256Ã—256` float32 grayscale in `[0,1]`,
output `logits`. Sigmoid is applied automatically server-side when needed.

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=DEVICE))
final = evaluate(val_loader)
print(json.dumps(final, indent=2))

onnx_path = OUT_DIR / 'brain_unet.onnx'
dummy = torch.randn(1, 1, CONFIG['SIZE'], CONFIG['SIZE'])
model.eval().cpu()
torch.onnx.export(
    model, dummy, str(onnx_path), opset_version=17,
    input_names=['input'], output_names=['logits'],
    dynamic_axes={'input': {0: 'batch', 2: 'h', 3: 'w'}, 'logits': {0: 'batch', 2: 'h', 3: 'w'}},
)

import onnx, onnxruntime as ort
onnx.checker.check_model(onnx.load(str(onnx_path)))
sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
with torch.no_grad():
    ref = torch.sigmoid(model(dummy)).numpy()
raw = sess.run(None, {'input': dummy.numpy()})[0]
prob = raw if (raw.min() >= 0 and raw.max() <= 1) else 1 / (1 + np.exp(-raw))
diff = float(np.abs(prob - ref).max())
print(f'ONNX vs PyTorch max |Î”|: {diff:.2e}')
assert diff < 1e-3, 'export parity check failed'
print(f'Done â†’ copy {onnx_path} to inference-service/models/brain_unet.onnx')
# On Colab: from google.colab import files; files.download(str(onnx_path))

## Next steps

1. Copy `outputs/brain_unet.onnx` into `inference-service/models/brain_unet.onnx`
   (Git LFS when pushing to a Hugging Face Space).
2. Set `MOCK_PREDICTIONS=0` once real weights are deployed.
3. Improve results honestly: more cases, full-volume training lists, deeper
   model, cross-validation â€” and report metrics with split discipline.

> Reminder: **this is a research prototype and not approved for clinical
> diagnosis.** Do not present outputs as diagnostic information.